In [1]:
import os
import gymnasium
import highway_env
import warnings
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

env_name = 'highway-v0'
root = f'{env_name}-PPO-C'

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ── Configuración ────────────────────────────────────────────────────────────
config = {
    "observation": {
        "type": "Kinematics",
        "vehicles_count": 15,
        "features": ["presence", "x", "y", "vx", "vy"],
        "features_range": {
            "x": [-100, 100],
            "y": [-100, 100],
            "vx": [-30, 30],
            "vy": [-30, 30],
        },
        "absolute": False,
        "order": "sorted",
        "normalize": True,
    },

    "action": {
        "type": "ContinuousAction",
    },

    "lanes_count": 4,
    "vehicles_count": 50,
    "vehicles_density": 1.2,
    "duration": 60,
    "initial_lane_id": None,

    "simulation_frequency": 15,
    "policy_frequency": 5,

    "collision_reward": -2.0,
    "high_speed_reward": 2.5,
    "lane_change_reward": -0.05,
    "right_lane_reward": 0.0,
    "reward_speed_range": [20, 30],
    "normalize_reward": True,

    "offroad_terminal": True,
    "controlled_vehicles": 1,
    "manual_control": False,
}

log_dir = f"{root}/logs/"
os.makedirs(log_dir, exist_ok=True)

# ── Entornos ─────────────────────────────────────────────────────────────────
env = make_vec_env(env_name, n_envs=8, monitor_dir=log_dir, env_kwargs={"config": config})

# ── Modelo (Carga o Creación) ──────────────────────────────────────────────────
TOTAL_TIMESTEPS = 200_000

print("Creando modelo desde cero...")
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    n_steps=256,
    batch_size=256,
    gae_lambda=0.95,
    gamma=0.99,
    n_epochs=10,
    learning_rate=3e-4,
    clip_range=0.2,
    ent_coef=0.05,
    policy_kwargs=dict(
        net_arch=[256, 256]
    ),
)

# ── Entrenamiento ─────────────────────────────────────────────────────────────
try:
    print("Entrenando al agente...")
    model.learn(total_timesteps=TOTAL_TIMESTEPS)
except KeyboardInterrupt:
    print("\nEntrenamiento interrumpido por el usuario (Ctrl+C). Guardando estado actual...")

finally:
    print("Guardando modelo...")
    model.save(f"{log_dir}/ppo_continuous_model")
    
    env.close()
    print("Entornos cerrados correctamente.")

<frozen importlib._bootstrap>:488: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.


Creando modelo desde cero...
Using cpu device
Entrenando al agente...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 3.97     |
|    ep_rew_mean     | 1.68     |
| time/              |          |
|    fps             | 10       |
|    iterations      | 1        |
|    time_elapsed    | 187      |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 6.34        |
|    ep_rew_mean          | 2.99        |
| time/                   |             |
|    fps                  | 10          |
|    iterations           | 2           |
|    time_elapsed         | 375         |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.049269002 |
|    clip_fraction        | 0.237       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.83       |
| 